## Import Libraries

In [1]:
import os

os.environ["HSA_OVERRIDE_GFX_VERSION"] = "10.3.0"

In [2]:
import numpy as np

import torch 
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import matplotlib.pyplot as plt

## Prepare train dataset

In [3]:
# Transform to turn into tensor and normalize
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307),(0.3081,)) # mean y std
])

In [4]:
# Load the dataset and normalize it
train_dataset = datasets.MNIST(
    root='./data',
    train=True,
    download=False,
    transform=transform
)

## Prepare test dataset

In [5]:
transform_test = transforms.ToTensor()

In [6]:
# Load the test dataset
test_dataset = datasets.MNIST(
    root='./data',
    train=False,
    download=False,
    transform=transform_test
)

## Data Loaders

In [7]:
# DataLoaders
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=32,
    shuffle=True,
    drop_last=True
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=len(test_dataset[0][0][0])
)

## Feed Forward Network 

In [8]:
# All variables
%whos

Variable         Type          Data/Info
----------------------------------------
DataLoader       type          <class 'torch.utils.data.dataloader.DataLoader'>
F                module        <module 'torch.nn.functio<...>/torch/nn/functional.py'>
datasets         module        <module 'torchvision.data<...>on/datasets/__init__.py'>
nn               module        <module 'torch.nn' from '<...>es/torch/nn/__init__.py'>
np               module        <module 'numpy' from '/ho<...>kages/numpy/__init__.py'>
os               module        <module 'os' (frozen)>
plt              module        <module 'matplotlib.pyplo<...>es/matplotlib/pyplot.py'>
test_dataset     MNIST         Dataset MNIST\n    Number<...>rm\nTransform: ToTensor()
test_loader      DataLoader    <torch.utils.data.dataloa<...>object at 0x76dd26d71f10>
torch            module        <module 'torch' from '/ho<...>kages/torch/__init__.py'>
train_dataset    MNIST         Dataset MNIST\n    Number<...>=(0.3081,))\n           )
t

In [9]:
import torch

print(f"¿GPU disponible?: {torch.cuda.is_available()}")
print(f"Nombre de la GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Ninguna'}")

¿GPU disponible?: True
Nombre de la GPU: AMD Radeon RX 6600


In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Entrenando en: {device}")

Entrenando en: cuda


In [11]:
def createTheMNISTNet():

    class mnistNet(nn.Module):
        def __init__(self):
            super().__init__()

            # input layer
            self.input = nn.Linear(784,64) # 784 pixels

            # hidden layer
            self.fc1 = nn.Linear(64,32)
            self.fc2 = nn.Linear(32,32)

            # output layer
            self.output = nn.Linear(32,10)

        def forward(self,x):
            x = x.view(x.shape[0], -1)
            
            x = F.relu( self.input(x) )
            x = F.relu( self.fc1(x) )
            x = F.relu( self.fc2(x) )
            return torch.log_softmax( self.output(x), dim=1 )

    net = mnistNet()

    lossfun = nn.NLLLoss()

    optimizer = torch.optim.SGD(net.parameters(), lr=0.1) # try with Adam later

    return net, lossfun, optimizer

In [12]:
net, lossfun, optimizer = createTheMNISTNet()

X,y = next(iter(train_loader))
yHat = net(X)

print(yHat)

loss = lossfun(yHat, y)
print(' ')
print('Loss:')
print(loss)

tensor([[-2.1989, -2.2519, -2.3355, -2.4028, -2.3985, -2.3224, -2.3767, -2.3727,
         -2.1067, -2.3022],
        [-2.1660, -2.2202, -2.3265, -2.3928, -2.4332, -2.2629, -2.3983, -2.4154,
         -2.1958, -2.2586],
        [-2.1646, -2.2160, -2.3595, -2.4056, -2.3977, -2.2779, -2.4233, -2.4001,
         -2.1933, -2.2332],
        [-2.1763, -2.2212, -2.3630, -2.4093, -2.3703, -2.2795, -2.4205, -2.4037,
         -2.1901, -2.2342],
        [-2.1897, -2.2211, -2.3496, -2.4091, -2.4213, -2.2819, -2.3968, -2.3980,
         -2.1382, -2.2674],
        [-2.2012, -2.1937, -2.3449, -2.3978, -2.3935, -2.2678, -2.3999, -2.3831,
         -2.2000, -2.2786],
        [-2.2270, -2.2033, -2.3378, -2.4088, -2.3889, -2.2640, -2.3982, -2.3760,
         -2.1856, -2.2697],
        [-2.1865, -2.2748, -2.3021, -2.3871, -2.4397, -2.2724, -2.3863, -2.4278,
         -2.1170, -2.2821],
        [-2.1942, -2.2095, -2.3354, -2.3852, -2.4204, -2.2581, -2.3936, -2.4156,
         -2.1816, -2.2725],
        [-2.1129, -

## Training

In [16]:
def function2trainTheModel():

    numepochs = 60

    net, lossfun, optimizer = createTheMNISTNet()
    net.to(device)

    losses = torch.zeros(numepochs)
    trainAcc = []
    testAcc = []

    for epochi in range(numepochs):

        # ===== Train =====
        batchAcc = []
        batchLoss = []
        for X,y in train_loader:

            X, y = X.to(device), y.to(device)
            
            # forward
            yHat = net(X)
            loss = lossfun(yHat, y)

            # back
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # loss
            batchLoss.append( loss.item() )

            # accuracy
            matches = torch.argmax(yHat, axis=1) == y # predictions
            matchesNumeric = matches.float() # to numeric
            accuracyPct = 100*torch.mean(matchesNumeric) # pct of mean
            batchAcc.append( accuracyPct.item() ) # add

        trainAcc.append( np.mean(batchAcc) )

        losses[epochi] = np.mean(batchLoss)

        # ===== Test =====
        X,y = next(iter(test_loader))
        X, y = X.to(device), y.to(device)
        yHat = net(X)

        acc = 100*torch.mean( (torch.argmax(yHat, axis=1)==y).float() )
        testAcc.append( acc.item() )
        
        print(f'Epoch: {epochi} \f trainAcc: {trainAcc[epochi]} \f testAcc: {testAcc[epochi]}')

    return trainAcc, testAcc, losses, net

In [ ]:
trainAcc, testAcc, losses, net = function2trainTheModel()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16,5))

ax[0].plot(losses)
ax[0].set_xlabel('Epochs')
ax[0].set_ylabel('Loss')
ax[0].set_ylim([0,3])
ax[0].set_title('Model loss')

ax[1].plot(trainAcc, label='Train')
ax[1].plot(testAcc, label='Test')
ax[1].set_xlabel('Epochs')
ax[1].set_ylabel('Accuracy (%)')
ax[1].set_ylim([10,100])
ax[1].set_title(f'Final model test accuracy: {testAcc[-1]:.2f}%')
ax[1].legend()